In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from tensorflow.keras.models import load_model
import os

print(" Imports ready")

2026-05-10 17:20:14.132838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports ready


In [2]:
# Paths
project_root = '/Users/bidisabiswas/PycharmProjects/Master-Thesis-AI-Robustness/thesis-ai-robustness'
repo_path = os.path.join(project_root, 'Stock-Price-Movement-Prediction')

# Load model
model = load_model(os.path.join(repo_path, 'best_enhanced_model.keras'))
print(f"✅ Model loaded")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# Load scaler
scaler = joblib.load(os.path.join(repo_path, 'artifacts', 'enhanced', 'feature_scaler_enhanced.pkl'))
print(f"✅ Scaler loaded")
print(f"   Expected features: {scaler.n_features_in_}")



✅ Model loaded
   Input shape: (None, 120, 24)
   Output shape: (None, 1)
✅ Scaler loaded
   Expected features: 24


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Load your simulated paths
normal_paths = np.load('../data/simulated/mc_returns_normal.npy')
crisis_2008_paths = np.load('../data/simulated/mc_returns_crisis_2008.npy')
crisis_covid_paths = np.load('../data/simulated/mc_returns_crisis_covid.npy')
synthetic_paths = np.load('../data/simulated/mc_returns_synthetic_extreme.npy')

print(f"Normal paths: {normal_paths.shape}")
print(f"2008 paths: {crisis_2008_paths.shape}")
print(f"COVID paths: {crisis_covid_paths.shape}")
print(f"Synthetic paths: {synthetic_paths.shape}")

Normal paths: (10000, 252)
2008 paths: (10000, 252)
COVID paths: (10000, 252)
Synthetic paths: (10000, 252)


In [7]:
def create_features(returns_paths, lookback=120):
    """
    Convert Monte Carlo returns to 24-feature format.
    
    Parameters:
    -----------
    returns_paths : array (n_paths, n_days)
        Monte Carlo returns
    lookback : int
        Number of days to look back (model expects 120)
    
    Returns:
    --------
    features : array (n_paths, lookback, 24)
    """
    n_paths, n_days = returns_paths.shape
    n_features = 24
    
    # Take last 'lookback' days
    returns_window = returns_paths[:, -lookback:]  # (n_paths, 120)
    
    # Create feature array
    features = np.zeros((n_paths, lookback, n_features))
    
    # Feature 0: Raw returns
    features[:, :, 0] = returns_window
    
    # Features 1-23: For now, use scaled versions of returns
    # This is simplified. In production, you'd add RSI, MACD, etc.
    for i in range(1, min(23, n_features-1)):
        # Different lags and transformations
        features[:, :, i] = returns_window * (0.9 ** i)  # Decaying weights
    
    return features

print("✅ Adapter function ready")

✅ Adapter function ready


In [9]:
def predict_scenario(model, scaler, paths, batch_size=500):
    """
    Run predictions on all paths.
    """
    n_paths = paths.shape[0]
    lookback = 120  # Model expects 120 timesteps
    n_features = 24
    
    all_predictions = []
    
    for i in range(0, n_paths, batch_size):
        batch = paths[i:i+batch_size]
        
        # Create features
        features = create_features(batch, lookback)
        
        # Reshape for scaler
        features_2d = features.reshape(-1, n_features)
        
        # Scale
        features_scaled = scaler.transform(features_2d)
        
        # Reshape back
        features_scaled = features_scaled.reshape(-1, lookback, n_features)
        
        # Predict
        preds = model.predict(features_scaled, verbose=0)
        all_predictions.append(preds)
        
        print(f"   Processed {min(i+batch_size, n_paths):,}/{n_paths:,} paths")
    
    return np.concatenate(all_predictions, axis=0)

print("✅ Prediction function ready")

✅ Prediction function ready


In [10]:
print("Running predictions on 10,000 paths per scenario...")
print("=" * 50)

print("\n1. Normal scenario...")
pred_normal = predict_scenario(model, scaler, normal_paths)

print("\n2. 2008 crisis scenario...")
pred_2008 = predict_scenario(model, scaler, crisis_2008_paths)

print("\n3. COVID crisis scenario...")
pred_covid = predict_scenario(model, scaler, crisis_covid_paths)

print("\n4. Synthetic extreme scenario...")
pred_synthetic = predict_scenario(model, scaler, synthetic_paths)

print("\n✅ All predictions complete!")
print(f"Normal predictions shape: {pred_normal.shape}")

Running predictions on 10,000 paths per scenario...

1. Normal scenario...
   Processed 500/10,000 paths
   Processed 1,000/10,000 paths
   Processed 1,500/10,000 paths
   Processed 2,000/10,000 paths
   Processed 2,500/10,000 paths
   Processed 3,000/10,000 paths
   Processed 3,500/10,000 paths
   Processed 4,000/10,000 paths
   Processed 4,500/10,000 paths
   Processed 5,000/10,000 paths
   Processed 5,500/10,000 paths
   Processed 6,000/10,000 paths
   Processed 6,500/10,000 paths
   Processed 7,000/10,000 paths
   Processed 7,500/10,000 paths
   Processed 8,000/10,000 paths
   Processed 8,500/10,000 paths
   Processed 9,000/10,000 paths
   Processed 9,500/10,000 paths
   Processed 10,000/10,000 paths

2. 2008 crisis scenario...
   Processed 500/10,000 paths
   Processed 1,000/10,000 paths
   Processed 1,500/10,000 paths
   Processed 2,000/10,000 paths
   Processed 2,500/10,000 paths
   Processed 3,000/10,000 paths
   Processed 3,500/10,000 paths
   Processed 4,000/10,000 paths
   P

In [11]:
def calculate_accuracy(predictions, actual_returns, lookback=120):
    """
    Calculate directional accuracy.
    """
    # Predicted direction (1 if prob > 0.5)
    pred_probs = predictions.flatten()
    pred_direction = (pred_probs > 0.5).astype(int)
    
    # Actual direction for the day AFTER lookback window
    actual_return = actual_returns[:, -(lookback+1)]
    actual_direction = (actual_return > 0).astype(int)
    
    # Accuracy
    correct = (pred_direction == actual_direction).sum()
    accuracy = correct / len(pred_direction) * 100
    
    return accuracy, pred_probs.mean()

# Calculate for each scenario
lookback = 120

acc_normal, prob_normal = calculate_accuracy(pred_normal, normal_paths, lookback)
acc_2008, prob_2008 = calculate_accuracy(pred_2008, crisis_2008_paths, lookback)
acc_covid, prob_covid = calculate_accuracy(pred_covid, crisis_covid_paths, lookback)
acc_synthetic, prob_synthetic = calculate_accuracy(pred_synthetic, synthetic_paths, lookback)

print("\n" + "=" * 50)
print("DIRECTIONAL ACCURACY RESULTS")
print("=" * 50)
print(f"Normal:         {acc_normal:.2f}% (avg prob: {prob_normal:.3f})")
print(f"2008 Crisis:    {acc_2008:.2f}% (avg prob: {prob_2008:.3f})")
print(f"COVID Crisis:   {acc_covid:.2f}% (avg prob: {prob_covid:.3f})")
print(f"Synthetic:      {acc_synthetic:.2f}% (avg prob: {prob_synthetic:.3f})")


DIRECTIONAL ACCURACY RESULTS
Normal:         49.26% (avg prob: 13599.702)
2008 Crisis:    49.26% (avg prob: 13599.824)
COVID Crisis:   49.26% (avg prob: 13599.754)
Synthetic:      49.26% (avg prob: 13599.893)


In [12]:
degradation_2008 = ((acc_normal - acc_2008) / acc_normal) * 100
degradation_covid = ((acc_normal - acc_covid) / acc_normal) * 100
degradation_synthetic = ((acc_normal - acc_synthetic) / acc_normal) * 100

print("\n" + "=" * 50)
print("ACCURACY DEGRADATION")
print("=" * 50)
print(f"2008 Crisis degradation:    {degradation_2008:.1f}%")
print(f"COVID Crisis degradation:   {degradation_covid:.1f}%")
print(f"Synthetic degradation:      {degradation_synthetic:.1f}%")


ACCURACY DEGRADATION
2008 Crisis degradation:    0.0%
COVID Crisis degradation:   0.0%
Synthetic degradation:      0.0%


In [13]:
# Quick test: Are your features different across scenarios?
features_normal = create_features(normal_paths[:10], lookback=120)
features_2008 = create_features(crisis_2008_paths[:10], lookback=120)

print(f"Normal features mean: {features_normal.mean():.6f}")
print(f"2008 features mean: {features_2008.mean():.6f}")
print(f"Are they different? {features_normal.mean() != features_2008.mean()}")

Normal features mean: 0.001276
2008 features mean: 0.002018
Are they different? True
